In [0]:
from pyspark.sql import functions as F

CATALOG = "workspace"
SCHEMA = "oss_ecosystem"

SILVER_CORE_TABLE = f"{CATALOG}.{SCHEMA}.silver_events_core"
SILVER_PUSH_TABLE = f"{CATALOG}.{SCHEMA}.silver_push_events"

In [0]:
silver_core_df = spark.table(SILVER_CORE_TABLE)
silver_push_df = spark.table(SILVER_PUSH_TABLE)

print("Silver Core rows:", silver_core_df.count())
print("Silver Push rows:", silver_push_df.count())

In [0]:
spark.sql(f"""
DESCRIBE DETAIL {SILVER_CORE_TABLE}
""").select(
    "numFiles",
    "sizeInBytes"
).show(truncate=False)

spark.sql(f"""
DESCRIBE DETAIL {SILVER_PUSH_TABLE}
""").select(
    "numFiles",
    "sizeInBytes"
).show(truncate=False)

In [0]:
repo_daily_activity_df = (
    silver_core_df
    .filter(F.col("repo_id").isNotNull())
    .withColumn(
        "event_date",
        F.to_date("event_ts")
    )
    .groupBy(
        "repo_id",
        "event_date"
    )
    .agg(
        F.count("*").alias("event_count"),
        F.countDistinct("actor_id").alias("active_contributors")
    )
)

In [0]:
repo_daily_activity_df.explain(
    mode="formatted"
)

In [0]:
import time

start_time = time.perf_counter()

result_count = repo_daily_activity_df.count()

elapsed_seconds = time.perf_counter() - start_time

print("Output rows:", result_count)
print("Execution time (seconds):", round(elapsed_seconds, 3))

In [0]:
repo_activity_distribution_df = (
    silver_core_df
    .filter(F.col("repo_id").isNotNull())
    .groupBy("repo_id")
    .agg(
        F.count("*").alias("event_count")
    )
)

repo_activity_distribution_df.orderBy(
    F.desc("event_count")
).show(20, truncate=False)

In [0]:
repo_activity_distribution_df.select(
    F.min("event_count").alias("min_events"),
    F.expr("percentile_approx(event_count, 0.5)").alias("median_events"),
    F.expr("percentile_approx(event_count, 0.95)").alias("p95_events"),
    F.expr("percentile_approx(event_count, 0.99)").alias("p99_events"),
    F.max("event_count").alias("max_events")
).show()

In [0]:
performance_baseline = {
    "silver_core_rows": 616687,
    "silver_core_files": 7,
    "silver_core_size_bytes": 10470961,
    "silver_push_files": 6,
    "silver_push_size_bytes": 30289668,
    "aggregation_output_rows": 73532,
    "aggregation_runtime_seconds": 0.559,
    "repo_event_median": 1,
    "repo_event_p95": 59,
    "repo_event_p99": 103,
    "repo_event_max": 393
}

performance_baseline

## Phase 9 conclusions

Measured baseline:

- Silver Core rows: 616,687
- Silver Core Delta files: 7
- Silver Core size: 10,470,961 bytes
- Silver Push Delta files: 6
- Silver Push size: 30,289,668 bytes
- Representative aggregation output rows: 73,532
- Representative aggregation runtime: 0.559 seconds

Execution-plan observations:

- The query was fully supported by Photon.
- Column pruning occurred and only required columns were scanned.
- A shuffle occurred for repository-level aggregation.
- The physical plan showed hash partitioning by repository ID.
- AdaptiveSparkPlan was present in the physical plan.

Skew observations:

- Median events per repository: 1
- P95: 59
- P99: 103
- Maximum: 393

Engineering decision:

No optimization is applied at the current scale.

The current tables have only a small number of Delta files, the representative aggregation is already fast, and no operationally significant skew was observed. Repartitioning, salting, caching, or compaction would therefore add complexity without evidence of benefit.

Performance should be reassessed after scaling the dataset to 24 hours and later to approximately 7 days.